<a href="https://colab.research.google.com/github/Oksana0020/Diamonds-game/blob/main/Diamonds_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Diamonds game
(Search tree)

Robinson Crusoe and Friday play diamonds game on desert island using two piles of stones.

At the beginning of the game, the player chooses to play as Robinson or Friday.
The other character is controlled by AI.

##Goal of the Game
Two piles of stones, Pile A and Pile B, are placed on the ground.

Players take turns removing stones.

The game ends when both piles become (0;0)

##Rules:

On each turn a player makes one move:

1. Remove 1 to 10 stones from one pile

2. Remove the same number from both piles from 1 to 5 stones

All moves must follow these limits.


##Ideas
I want to introduce piles evolution gradually

first: small fixed piles which will be fast to test

then: (100, 100)

then: randomized piles from 80–120, generated once at game start and shown to player.

Also each player can receive two Diamond Tokens like special tools found on the island.A player may spend one token to perform a special move (for example this move can mean - remove the same number of stones from both piles 6-10)

Lucky 13 Rule - When Lucky 13 is triggered 5 stones are added to both piles, triggers multiple times during a game, may benefit or harm a player

Life on the island is unpredictable.From time to time, storms affect the piles. A storm occurs after both players have taken 5 turns each (10 total moves, then 20,30...).Storms repeats regularly.When a storm happens 3 stones are added to one random pile

If time allows to add Twisted Mode
Normal Mode:
The player who makes the move that reaches (0, 0) wins

Twisted Mode:
The player who makes the move that reaches (0, 0) loses

Possible add some dialogues in game - each character has a personality and speaks after making moves like commenting moves

Also need to display pile bars and token counts

##Constants and character labels

In [18]:
WIN_VALUE = 1
LOSS_VALUE = -1

CHAR_ROBINSON = "Robinson"
CHAR_FRIDAY = "Friday"

##CurrentBoard with initial rules

The game state is represented by two piles of stones:

a = first pile

b = second pile

The piles are position-based, so their order is preserved.

Base legal moves:
*   Remove 1-10 stones from one pile
*   Remove 1-5 stoens from both piles (same number)

Branching factor = number of possible moves from a position

In [19]:
class CurrentBoard:
    def __init__(self, a=5, b=10):
      self.a = int(a)
      self.b = int(b)
      self.state = self.state_of_play()

    def display_board(self, game_display=False):
        print(f"Piles: ({self.a}, {self.b})")

    def state_of_play(self):
        return "FINISHED" if (self.a == 0 and self.b == 0) else "PLAYING"

    def other_player(self, forPlayer):
        return "O" if forPlayer == "X" else "X"

    def all_possible_moves(self, forPlayer="X"):
        moves = []

        # remove from pile a (max 10)
        for k in range(1, min(10, self.a) + 1):
            moves.append(CurrentBoard(self.a - k, self.b))

        # remove from pile b (max 10)
        for k in range(1, min(10, self.b) + 1):
            moves.append(CurrentBoard(self.a, self.b - k))

        # remove same from both (max 5)
        for k in range(1, min(5, self.a, self.b) + 1):
            moves.append(CurrentBoard(self.a - k, self.b - k))

        return moves

##Checking count of moves and sample results.

In [20]:
cb = CurrentBoard(5, 10)
cb.display_board()
kids = cb.all_possible_moves()
print("Number of possible moves:", len(kids))
print([(k.a, k.b) for k in kids])

Piles: (5, 10)
Number of possible moves: 20
[(4, 10), (3, 10), (2, 10), (1, 10), (0, 10), (5, 9), (5, 8), (5, 7), (5, 6), (5, 5), (5, 4), (5, 3), (5, 2), (5, 1), (5, 0), (4, 9), (3, 8), (2, 7), (1, 6), (0, 5)]


To let the AI think ahead, I build search tree.

Each node of the tree stores:

the current board position

the player whose turn it is

all possible child positions, meaning all legal moves from that board

**Terminal position**

A terminal position is a leaf of the tree.

In this game, the terminal board is(0, 0)

At (0,0) no legal moves remain, so the player whose turn it is loses.

**Minimax idea**

The tree is evaluated using the minimax algorithm.

The main idea is:

the current player wants the best possible result

the opponent also plays optimally

so the algorithm explores future moves and works backward from the end of the game

**Ply depth**

The implementation uses ply depth, which means the move number in the tree.

even depth → maximizing level

odd depth → minimizing level

So the algorithm alternates between the two players as it moves down the tree.

**Values at terminal nodes**

There is no draw in this game, only win or loss.

So terminal nodes are assigned values:

-1 means loss

+1 means win

If a terminal position is reached and it is the current player’s turn, that player has no moves, so that position is a loss for that player.

**Recursive evaluation**

For non-terminal nodes:

all children are evaluated recursively

children are sorted by their minimax values

the node then chooses:

the maximum value at even depth (choose highest value)

the minimum value at odd depth (choose lowest value)





+1 means the current player can force a win

-1 means the current player will lose if the opponent also plays optimally

A position is winning if there exists at least one move that leads to a losing position for the opponent.Otherwise the position is losing.

SearchTreeNode class builds a tree of all possible future moves and uses minimax to decide whether a position is winning or losing.

if playing -look at all possible next positions and create branches for them
(generate all possible moves,create a child node for each move,switch player,
go one level deeper (ply + 1)

if finished - If it’s my turn and I have no moves, I lose otherwise, I win

In [21]:
class SearchTreeNode:
    def __init__(self, current_board, forPlayer, ply=0):
        self.board = current_board
        self.ply_depth = ply
        self.value_is_assigned = False
        self.children = []
        # expand children if game is not finished
        if current_board.state == "PLAYING":
            for move in current_board.all_possible_moves(forPlayer):
                self.children.append(
                    SearchTreeNode(move, self.board.other_player(forPlayer), self.ply_depth + 1)
                )
        else:
            self.value_is_assigned = True
            # no draw,only win or loss
            if self.ply_depth % 2 == 0:
                self.value = -1
            else:
                self.value = 1

    def min_max_value(self):
        if self.value_is_assigned:
            return self.value
        # evaluate children first
        self.children = sorted(self.children, key=lambda x: x.min_max_value())
        self.value_is_assigned = True
        # max player at even ply, min at odd ply
        if self.ply_depth % 2 == 0:
            self.value = self.children[-1].value
        else:
            self.value = self.children[0].value
        return self.value

At even depths, the algorithm selects the maximum value, and at odd depths it selects the minimum value, representing alternating turns of two optimal players.

If it's your turn (even depth):

pick the best move (maximum)

If it's opponent turn (odd depth):

assume they pick the worst for AI (minimum)

##Checking minimax value on a small starting position

In [22]:
cb = CurrentBoard(5, 10)
st = SearchTreeNode(cb, "X")

print("Root board:", (cb.a, cb.b))
print("Children count:", len(st.children))
print("Minimax value:", st.min_max_value())

Root board: (5, 10)
Children count: 20
Minimax value: 1


##Choose the best child

The idea is if a child has value -1, that means the opponent loses in that position so moving to that child is a winning move for the current player

implementation:

all children are evaluated using minimax

children are sorted by their minimax values

the best move is the one with the maximum value

In [23]:
def best_child_board(node: SearchTreeNode):
    if not node.children:
        return None
    node.children = sorted(node.children, key=lambda x: x.min_max_value())
    return node.children[-1].board

##Checking best move selection

Testing the function on a small starting position.

In [24]:
cb = CurrentBoard(5, 10)
st = SearchTreeNode(cb, "X")
print("Root board:", (cb.a, cb.b))
print("Children count:", len(st.children))
print("Minimax value:", st.min_max_value())
best = best_child_board(st)
print("Best next board:", (best.a, best.b) if best else None)

Root board: (5, 10)
Children count: 20
Minimax value: 1
Best next board: (5, 3)


##Validate player moves
Before creating a playable loop I need a helper that checks whether a move is legal

generate all possible next boards

check if (a,b) appears among them

In [25]:
def is_legal_move(current: CurrentBoard, nxt: CurrentBoard) -> bool:
    return any((m.a, m.b) == (nxt.a, nxt.b) for m in current.all_possible_moves())

testing one legal and one illegal move from the same starting position.

In [26]:
cb = CurrentBoard(5, 8)
legal_test = CurrentBoard(5, 3)
illegal_test = CurrentBoard(1, 1)

print("Legal move test:", is_legal_move(cb, legal_test))
print("Illegal move test:", is_legal_move(cb, illegal_test))

Legal move test: True
Illegal move test: False


##Player move input

The player enters a move and the program converts it into new board state.

Move format: A-pile A, B-pile B, AB-both, n-number

A n - remove number stones from pile A

B n - remove number stones from pile B

AB n - remove number stones from both piles

In [27]:
def apply_player_move(current_board: CurrentBoard, move_text: str) -> CurrentBoard:
    parts = move_text.strip().upper().replace(",", " ").split()
    if len(parts) != 2:
        raise ValueError("Move must look like: A 3, B 4 or AB 2")
    move_type, amount_text = parts
    amount = int(amount_text)
    if move_type == "A":
        return CurrentBoard(current_board.a - amount, current_board.b)
    if move_type == "B":
        return CurrentBoard(current_board.a, current_board.b - amount)
    if move_type == "AB":
        return CurrentBoard(current_board.a - amount, current_board.b - amount)
    raise ValueError("Move type must be A, B or AB")

##first playable version of game

The game loop will allow:

human player to enter a move with input

AI to respond using minimax algorithm

Players take turns until the game reaches a terminal state (0;0).

Maximum depth = total number of stones = 16

number of moves decreases as the piles get smaller, so total number of nodes is thousands at the beginning and then get smaller

In [28]:
def count_nodes(node):
    total = 1
    for child in node.children:
        total += count_nodes(child)
    return total

cb = CurrentBoard(5, 11)
st = SearchTreeNode(cb, "X")

print("Total nodes:", count_nodes(st))

Total nodes: 15931244


In [29]:
def play_game_basic(start_a=5, start_b=11):
    cb = CurrentBoard(start_a, start_b)
    current_player = "X"   # human starts

    print("Diamonds Game - basic version")
    print("You are X, AI is O")
    print("Make your move choosing one of the options:")
    print("- A n  -> remove n stones from pile A")
    print("- B n  -> remove n stones from pile B")
    print("- AB n -> remove n stones from both piles")
    print()

    while True:
        print("-------------------")
        print("Player to move:", current_player)
        cb.display_board()

        if cb.state == "FINISHED":
            winner = "O (AI)" if current_player == "X" else "X (Human)"
            print("Winner:", winner)
            return

        if current_player == "X":
            raw = input("Enter your move: ")
            cb = apply_player_move(cb, raw)
            current_player = "O"

        else:
            st = SearchTreeNode(cb, "O")
            st.min_max_value()
            nxt = best_child_board(st)

            if nxt is None:
                print("Winner: X (Human)")
                return

            print("AI chooses:", (nxt.a, nxt.b))
            cb = nxt
            current_player = "X"

In [ ]:
play_game_basic(5, 11)

Diamonds Game - basic version
You are X, AI is O
Make your move choosing one of the options:
- A n  -> remove n stones from pile A
- B n  -> remove n stones from pile B
- AB n -> remove n stones from both piles

-------------------
Player to move: X
Piles: (5, 11)
Enter your move: B 5
-------------------
Player to move: O
Piles: (5, 6)
AI chooses: (1, 2)
-------------------
Player to move: X
Piles: (1, 2)
Enter your move: AB 1
-------------------
Player to move: O
Piles: (0, 1)
AI chooses: (0, 0)
-------------------
Player to move: X
Piles: (0, 0)
Winner: O (AI)


##Adding validation for player input

Validation includes:

the move format is correct
the number of stones is valid
the move follows game rules
piles do not go below zero

If an invalid move is entered, program shows error message and asks for input again

Updating player move function

In [30]:
def apply_player_move(current_board: CurrentBoard, move_text: str) -> CurrentBoard:
    parts = move_text.strip().upper().replace(",", " ").split()

    if len(parts) != 2:
        raise ValueError("Use format: A 3, B 4, or AB 2")
    move_type, amount_text = parts
    try:
        amount = int(amount_text)
    except ValueError:
        raise ValueError("The number of stones must be an integer")

    if amount <= 0:
        raise ValueError("The number of stones must be greater than 0")

    if move_type == "A":
        if amount > 10:
            raise ValueError("From one pile you can remove at most 10 stones")
        if amount > current_board.a:
            raise ValueError("Pile A does not have that many stones")
        return CurrentBoard(current_board.a - amount, current_board.b)

    if move_type == "B":
        if amount > 10:
            raise ValueError("From one pile you can remove at most 10 stones")
        if amount > current_board.b:
            raise ValueError("Pile B does not have that many stones")
        return CurrentBoard(current_board.a, current_board.b - amount)

    if move_type == "AB":
        if amount > 5:
            raise ValueError("From both piles you can remove at most 5 stones")
        if amount > current_board.a or amount > current_board.b:
            raise ValueError("Both piles must have enough stones for this move")
        return CurrentBoard(current_board.a - amount, current_board.b - amount)

    raise ValueError("Move type must be A, B, or AB")

Updating play loop

In [33]:
def play_game_basic(start_a=5, start_b=11):
    cb = CurrentBoard(start_a, start_b)
    current_player = "X"   # human starts

    print("Diamonds Game - basic version")
    print("You are X, AI is O")
    print("Make your move choosing one of the options:")
    print("- A n  -> remove n stones from pile A")
    print("- B n  -> remove n stones from pile B")
    print("- AB n -> remove n stones from both piles")
    print()

    while True:
        print("-------------------")
        print("Player to move:", current_player)
        cb.display_board()

        if cb.state == "FINISHED":
            winner = "O (AI)" if current_player == "X" else "X (Human)"
            print("Winner:", winner)
            return

        if current_player == "X":
            while True:
                try:
                    raw = input("Enter your move: ")
                    cb = apply_player_move(cb, raw)
                    break
                except ValueError as e:
                    print("Invalid move:", e)

            current_player = "O"

        else:
            st = SearchTreeNode(cb, "O")
            st.min_max_value()
            nxt = best_child_board(st)

            if nxt is None:
                print("Winner: X (Human)")
                return

            print("AI chooses:", (nxt.a, nxt.b))
            cb = nxt
            current_player = "X"

In [34]:
play_game_basic(5, 11)

Diamonds Game - basic version
You are X, AI is O
Make your move choosing one of the options:
- A n  -> remove n stones from pile A
- B n  -> remove n stones from pile B
- AB n -> remove n stones from both piles

-------------------
Player to move: X
Piles: (5, 11)
Enter your move: A 11
Invalid move: From one pile you can remove at most 10 stones
Enter your move: B 44
Invalid move: From one pile you can remove at most 10 stones
Enter your move: A -2
Invalid move: The number of stones must be greater than 0
Enter your move: A 5
-------------------
Player to move: O
Piles: (0, 11)
AI chooses: (0, 1)
-------------------
Player to move: X
Piles: (0, 1)
Enter your move: B1
Invalid move: Use format: A 3, B 4, or AB 2
Enter your move: B 1
-------------------
Player to move: O
Piles: (0, 0)
Winner: X (Human)


##Tested the following wrong variants
Enter your move: A 20
Invalid move: From one pile you can remove at most 10 stones

Enter your move: AB 6
Invalid move: From both piles you can remove at most 5 stones

Enter your move: B 100
Invalid move: From one pile you can remove at most 10 stones

Enter your move: A 9
Invalid move: Pile A does not have that many stones



simple dialogues, bars with current state of piles, character selection